# Mini-Project 2: Uganda Tax Engine

This notebook demonstrates a Uganda tax calculator using abstraction, composition, custom exceptions, type hints, and a Decimal-only `Money` value object.


In [1]:
from datetime import date
from decimal import Decimal
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

from src.tax_engine import (
    CorporateIncomeTax, Invoice, InvoiceDirection, Money, PAYE,
    PayrollService, SupplyTreatment, TaxContext, TaxPlanner, VAT,
)

EFFECTIVE_DATE = date(2026, 7, 1)
rng = np.random.default_rng(2026)

## Regulatory baseline and assumptions

The engine uses the URA PAYE schedule effective **1 July 2026**, confirmed against the URA PAYE page and its 2026 amendment notice. Resident and non-resident monthly schedules are both supported. NSSF is 5% employee plus 10% employer on gross monthly wage. The VAT standard rate is 18%, and resident-company income tax is 30% of chargeable income. Local Service Tax is configurable.


## Money value object

`Money` accepts `Decimal`, integer, or string input, uses UGX only, rounds to a whole shilling with `ROUND_HALF_UP`, and overloads addition, multiplication, less-than, and representation. Passing a float is intentionally rejected.


In [3]:
salary = Money("4500000")
allowance = Money("300000")
print("Salary:", salary)
print("Salary plus allowance:", salary + allowance)
print("Five percent:", salary * Decimal("0.05"))
print("Comparison:", Money(1000) < Money(2000))
print("Representation:", repr(salary))

try:
    Money(0.1)
except TypeError as exc:
    print("Float protection:", exc)

Salary: UGX 4,500,000
Salary plus allowance: UGX 4,800,000
Five percent: UGX 225,000
Comparison: True
Representation: Money(UGX 4,500,000)
Float protection: float is not accepted for money; use Decimal, int, or str


## PAYE calculator

In [4]:
paye = PAYE()
edges = [335_000, 335_001, 410_000, 485_000, 10_000_000, 10_000_001]
boundary_rows = []
for income in edges:
    resident_tax = paye.calculate(TaxContext(income=Money(income), on_date=EFFECTIVE_DATE)).amount
    nonresident_tax = paye.calculate(
        TaxContext(income=Money(income), resident=False, on_date=EFFECTIVE_DATE)
    ).amount
    boundary_rows.append({
        "Monthly income (UGX)": income,
        "Resident PAYE (UGX)": int(resident_tax.amount),
        "Non-resident PAYE (UGX)": int(nonresident_tax.amount),
    })
boundary_table = pd.DataFrame(boundary_rows)
display(boundary_table)

,Monthly income (UGX),Resident PAYE (UGX),Non-resident PAYE (UGX)
0,335000,0,33500
1,335001,0,33500
2,410000,15000,48500
3,485000,33750,71000
4,10000000,2888250,2925500
5,10000001,2888250,2925500


## Payroll pipeline

The pipeline first combines salary and cash allowances. NSSF is calculated on that gross amount. The configured LST deduction is applied before PAYE as required by the assignment; PAYE is then calculated on gross less LST. Net pay deducts employee NSSF, LST, and PAYE. The employer's 10% NSSF contribution is reported as an employer cost and is never deducted from the employee's pay.


In [6]:
payroll = PayrollService(lst=Money(10_000))
payroll_result = payroll.calculate(
    Money(4_500_000), Money(300_000), resident=True, on_date=EFFECTIVE_DATE
)
payroll_table = pd.DataFrame([
    {"Item": "Gross cash pay", "UGX": payroll_result.gross_cash_pay.amount},
    {"Item": "Employee NSSF (deduction)", "UGX": payroll_result.employee_nssf.amount},
    {"Item": "Employer NSSF (employer cost)", "UGX": payroll_result.employer_nssf.amount},
    {"Item": "Local Service Tax", "UGX": payroll_result.local_service_tax.amount},
    {"Item": "PAYE", "UGX": payroll_result.paye.amount},
    {"Item": "Net pay", "UGX": payroll_result.net_pay.amount},
])
display(payroll_table)

,Item,UGX
0,Gross cash pay,4800000
1,Employee NSSF (deduction),240000
2,Employer NSSF (employer cost),480000
3,Local Service Tax,10000
4,PAYE,1325250
5,Net pay,3224750


## VAT module

Invoice amounts are VAT-exclusive. Standard-rated outputs create output VAT and standard-rated purchases create input VAT. Zero-rated supplies have a 0% output rate, while exempt supplies are outside the charge in this simplified assignment model. A negative balance is reported as credit carried forward rather than a negative payable amount.


In [7]:
invoices = [
    Invoice("SALE-001", Money(5_000_000), SupplyTreatment.STANDARD, InvoiceDirection.OUTPUT),
    Invoice("EXPORT-001", Money(2_000_000), SupplyTreatment.ZERO_RATED, InvoiceDirection.OUTPUT),
    Invoice("EXEMPT-001", Money(500_000), SupplyTreatment.EXEMPT, InvoiceDirection.OUTPUT),
    Invoice("PURCHASE-001", Money(1_500_000), SupplyTreatment.STANDARD, InvoiceDirection.INPUT),
]
vat_result = VAT().calculate(TaxContext(metadata={"invoices": invoices}))
print("Output VAT:", vat_result.details["output_vat"])
print("Input VAT:", vat_result.details["input_vat"])
print("Net VAT payable:", vat_result.amount)
print("Credit carried forward:", vat_result.details["credit_carried_forward"])

Output VAT: UGX 900,000
Input VAT: UGX 270,000
Net VAT payable: UGX 630,000
Credit carried forward: UGX 0


## Corporate income tax

Chargeable income is revenue less allowable deductions, floored at zero. The engine then applies the 30% resident-company rate.


In [8]:
corporate = CorporateIncomeTax().calculate(
    TaxContext(income=Money(100_000_000), deductions=Money(40_000_000))
)
print("Chargeable income:", corporate.details["chargeable_income"])
print("Corporate income tax:", corporate.amount)


Chargeable income: UGX 60,000,000
Corporate income tax: UGX 18,000,000


## Extension B - TaxPlanner

`TaxPlanner` compares two payroll structures. It composes `PayrollService`; it does not subclass a tax because planning is a coordination responsibility rather than a new tax rule.


In [10]:
planner = TaxPlanner(payroll)
comparison = planner.compare(
    (Money(4_500_000), Money(300_000)),
    (Money(4_800_000), Money(0)),
    on_date=EFFECTIVE_DATE,
)
print("Net-pay difference (scenario B minus A):", comparison["net_pay_difference_b_minus_a"])


Net-pay difference (scenario B minus A): UGX 0


## Testing and reproducibility

The repository's pytest suite contains the required PAYE boundary tests and Hypothesis properties. The notebook repeats representative deterministic checks, including 250 seeded random incomes, so `Restart & Run All` provides immediate evidence even before pytest is invoked.


In [12]:
expected_resident = {
    335_000: 0,
    335_001: 0,
    410_000: 15_000,
    485_000: 33_750,
    10_000_000: 2_888_250,
    10_000_001: 2_888_250,
}
for income, expected in expected_resident.items():
    actual = paye.calculate(TaxContext(income=Money(income), on_date=EFFECTIVE_DATE)).amount
    assert actual == Money(expected)

samples = sorted(int(x) for x in rng.integers(0, 100_000_001, size=250))
taxes = [paye.calculate(TaxContext(income=Money(x), on_date=EFFECTIVE_DATE)).amount for x in samples]
assert all(Money.zero() <= tax <= Money(income) for income, tax in zip(samples, taxes))
assert all(left <= right for left, right in zip(taxes, taxes[1:]))
assert payroll_result.net_pay == (
    payroll_result.gross_cash_pay - payroll_result.employee_nssf
    - payroll_result.local_service_tax - payroll_result.paye
)
print("Boundary, non-negative, bounded, monotonic, and payroll reconciliation checks passed.")


Boundary, non-negative, bounded, monotonic, and payroll reconciliation checks passed.


# Discussion

## 1. Why `float` is unsafe for tax computation

Binary floating-point represents numbers as sums of powers of two. Many ordinary decimal fractions have no finite binary representation. For example, Python evaluates `0.1 + 0.2` as approximately `0.30000000000000004`, not an exact decimal 0.3. A single discrepancy is small, but tax and payroll systems repeatedly multiply rates, add components, compare band boundaries, and round. If the moment and direction of rounding are not explicit, two systems can disagree by a shilling; across thousands of employees or invoices those differences accumulate and make reconciliation difficult.

`Decimal` stores base-ten values and lets the program specify the legal or organisational rounding rule. The `Money` class accepts `Decimal`, integer, or string input and refuses floats so a binary approximation cannot enter unnoticed. It quantises every result to one UGX using `ROUND_HALF_UP`. This policy is visible, centralised, and testable. A production system may require different statutory rules for particular taxes, but those rules should still be explicit rather than delegated to the incidental behaviour of `round(float_value)`.

A concrete example is a rate calculation on repeated line items. With floats, `sum([0.1] * 10)` is not exactly 1.0. With `Decimal("0.1")`, the sum is exactly `Decimal("1.0")`. Similarly, computing 18% VAT from a decimal invoice amount should begin with `Decimal("0.18")`, not `0.18`, and round at the legally required stage. The project therefore rejects `Money(0.1)` with a `TypeError`.

## 2. Where inheritance is justified, and where it is avoided

Inheritance is justified at the behavioural boundary shared by all calculators. The abstract `Tax` class states that a calculator accepts a `TaxContext` and returns a `TaxResult`. PAYE, NSSF, LST, VAT, and corporate income tax can therefore be invoked through a common interface while keeping their algorithms separate. This is useful polymorphism: a caller can depend on the contract rather than a concrete class. The custom exception hierarchy is another appropriate use because callers may catch all `TaxError` cases or handle an `InvalidIncomeError` and `NoScheduleForDateError` differently.

Inheritance was deliberately avoided for data values and workflow coordination. `Money` is a value object, not a kind of tax. An `Invoice` is data, not a specialised `TaxContext`. `PayrollService` does not inherit from PAYE or NSSF because payroll *has* those calculators and coordinates them; it is not itself a subtype of either. Composition makes the calculation order visible and lets a different LST or PAYE implementation be substituted without a fragile inheritance tree. `TaxPlanner` similarly composes `PayrollService` because comparison is a service applied to results, not another statutory charge.

Deep class hierarchies would hide which rule is active and tempt subclasses to depend on internal state. This design keeps classes small: values hold validated data, individual tax classes own one rule set, and services orchestrate them. The result satisfies the OOP requirement without using inheritance where a simple object relationship is clearer.

## Limitations

Tax rules change, and this engine contains only the schedule effective 1 July 2026. It intentionally raises `NoScheduleForDateError` for earlier dates rather than silently applying the wrong law. The assignment's simplified VAT model does not decide whether every purchase is creditable, handle partial exemption, or prepare a statutory return. LST is a configured amount instead of a complete local-authority schedule. A production engine would require formally reviewed legislation, effective-dated tables, taxpayer-specific rules, secure audit logs, and approval by a qualified Ugandan tax professional.
